<a href="https://colab.research.google.com/github/DarkLyng/Proyecto_Modelos/blob/main/Proyecto_Modelos_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Preparar datos**

In [61]:
#Bibliotecas de Python

import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import csv

warnings.filterwarnings('ignore')

In [62]:
xls = pd.ExcelFile("/content/Contoso Sales.xlsx")
print(xls.sheet_names)

['DimChannel', 'DimGeography', 'DimProduct', 'DimProductCategory', 'DimProductSubCategory', 'DimPromotion', 'DimStores']


In [63]:
tabs = ['DimChannel', 'DimGeography', 'DimProduct', 'DimProductCategory', 'DimProductSubCategory', 'DimPromotion', 'DimStores']
tablas = [] # Secambio [] a {}

In [64]:
for tab in tabs:
  df = pd.read_excel("/content/Contoso Sales.xlsx", sheet_name=tab)
  tablas.append(df)

In [27]:
#for tab in tabs:
#  tablas[tab] = pd.read_excel("/content/Contoso Sales.xlsx", sheet_name=tab)

In [26]:
#tablas['FactSales'] = pd.read_csv("FactSales.txt", sep='\t')

#print("Tablas cargadas exitosamente:", list(tablas.keys()))

In [65]:
ventas = pd.read_csv("FactSales.txt", sep="\t")
tablas.append(ventas)


In [73]:
print(f"Total de tablas cargadas en la lista: {len(tablas)}")

Total de tablas cargadas en la lista: 8


In [75]:
#Comprobar por separado el número de columnas y filas
tablas[2].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1690 entries, 0 to 1689
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ProductKey             1690 non-null   int64  
 1   ProductName            1690 non-null   object 
 2   ProductDescription     1690 non-null   object 
 3   ProductSubcategoryKey  1690 non-null   int64  
 4   Manufacturer           1690 non-null   object 
 5   BrandName              1690 non-null   object 
 6   ClassID                1690 non-null   int64  
 7   ClassName              1690 non-null   object 
 8   ColorID                1690 non-null   int64  
 9   ColorName              1690 non-null   object 
 10  Size                   1240 non-null   object 
 11  Weight                 1526 non-null   float64
 12  UnitCost               1690 non-null   float64
 13  UnitPrice              1690 non-null   float64
dtypes: float64(3), int64(4), object(7)
memory usage: 185.0+ 

In [78]:
print("* Conteo de filas, columnas y datos fantantes *\n")

for i, df in enumerate(tablas):
    # 1. Filas y Columnas
    filas = df.shape[0]
    columnas = df.shape[1]

    # 2. Total de celdas vacías en toda la tabla
    celdas_vacias_tot = df.isnull().sum().sum()

    # Intentamos recuperar el nombre original usando la lista 'tabs' que creaste antes
    nombre_tabla = tabs[i] if i < len(tabs) else "FactSales"

    print(f"Índice [{i}] - Tabla: '{nombre_tabla}'")
    print(f"  Filas: {filas}")
    print(f"  Columnas: {columnas}")
    print(f"  Total de celdas vacías: {celdas_vacias_tot}")
    print("-" * 45)

* Conteo de filas, columnas y datos fantantes *

Índice [0] - Tabla: 'DimChannel'
  Filas: 4
  Columnas: 2
  Total de celdas vacías: 0
---------------------------------------------
Índice [1] - Tabla: 'DimGeography'
  Filas: 674
  Columnas: 3
  Total de celdas vacías: 3
---------------------------------------------
Índice [2] - Tabla: 'DimProduct'
  Filas: 1690
  Columnas: 14
  Total de celdas vacías: 614
---------------------------------------------
Índice [3] - Tabla: 'DimProductCategory'
  Filas: 8
  Columnas: 2
  Total de celdas vacías: 0
---------------------------------------------
Índice [4] - Tabla: 'DimProductSubCategory'
  Filas: 44
  Columnas: 3
  Total de celdas vacías: 0
---------------------------------------------
Índice [5] - Tabla: 'DimPromotion'
  Filas: 28
  Columnas: 7
  Total de celdas vacías: 0
---------------------------------------------
Índice [6] - Tabla: 'DimStores'
  Filas: 310
  Columnas: 6
  Total de celdas vacías: 1
---------------------------------------

In [81]:
def analizar_tabla_lista(df, nombre_tabla):
    print("\n" + "="*60)
    print(f"ANÁLISIS DE VARIABLES DE LA TABLA: {nombre_tabla}")
    print("="*60)

    for col in df.columns:
        dtype = df[col].dtype
        total_valores = len(df[col])
        valores_unicos = df[col].nunique()
        faltantes = df.isnull().sum().sum()

        print(f"\n🔹 Columna: '{col}' | Tipo original: {dtype}")
        print(f"  - Datos faltantes: {faltantes} ({ (faltantes/total_valores)*100:.2f}%)")
        print(f"  - Cardinalidad (Valores únicos): {valores_unicos}")

        # Variables Numéricas Continua o Cuantitativa (Media, Varianza, Inclinación, Kurtosis)
        if np.issubdtype(dtype, np.number) and valores_unicos > 10:
            print("  - Clasificación sugerida: Variable Numérica Continua o Cuantitativa")

            media = df[col].mean()
            mediana = df[col].median()
            desv_est = df[col].std()
            varianza = df[col].var()
            inclinacion = df[col].skew()   # Inclinación (Skewness)
            curtosis = df[col].kurt()      # Kurtosis

            print(f"  - Valor Esperado (Media): {media:.4f}")
            print(f"  - Mediana: {mediana:.4f}")
            print(f"  - Dispersión -> Varianza: {varianza:.4f} | Desv. Estándar: {desv_est:.4f}")
            print(f"  - Forma -> Inclinación (Skewness): {inclinacion:.4f} | Kurtosis: {curtosis:.4f}")

            if abs(inclinacion) < 0.5:
                print("  - Interpretación: Distribución relativamente simétrica.")
            else:
                print(f"  - Interpretación: Sesgada a la {'derecha' if inclinacion > 0 else 'izquierda'}.")

        # Variables Cualitativas / Discretas (Frecuencias y Probabilidad Empírica)
        else:
            print("  - Clasificación sugerida: Variable Cualitativa / Discreta")
            frecuencias = df[col].value_counts(dropna=False).head(5)
            probabilidades = df[col].value_counts(dropna=False, normalize=True).head(5)

            print("  - Top 5 valores con mayor Frecuencia y Probabilidad Empírica P(X=x):")
            for val, freq in frecuencias.items():
                prob = probabilidades[val]
                print(f"    * Valor: {val} | Frecuencia: {freq} | Probabilidad: {prob:.4f} ({prob*100:.2f}%)")

In [51]:
# 3. EJECUCIÓN DEL ANÁLISIS POR ÍNDICES
# ==========================================

# Analizar la tabla DimProduct (Índice 2)
analizar_tabla_lista(tablas[2], "DimProduct")

# Analizar la tabla FactSales (Último índice de la lista)
analizar_tabla_lista(tablas[-1], "FactSales")


ANÁLISIS DE VARIABLES DE LA TABLA: DimProduct

🔹 Columna: 'ProductKey' | Tipo original: int64
  - Datos faltantes: 0 (0.00%)
  - Cardinalidad (Valores únicos): 1690
  - Clasificación sugerida: Variable Numérica Continua o Cuantitativa
  - Valor Esperado (Media): 860.1805
  - Mediana: 845.5000
  - Dispersión -> Varianza: 274465.4766 | Desv. Estándar: 523.8945
  - Forma -> Inclinación (Skewness): 0.4078 | Kurtosis: -0.0250
  - Interpretación: Distribución relativamente simétrica.

🔹 Columna: 'ProductName' | Tipo original: object
  - Datos faltantes: 0 (0.00%)
  - Cardinalidad (Valores únicos): 1690
  - Clasificación sugerida: Variable Cualitativa / Discreta
  - Top 5 valores con mayor Frecuencia y Probabilidad Empírica P(X=x):
    * Valor: Contoso In-Line Coupler E180 Black | Frecuencia: 1 | Probabilidad: 0.0006 (0.06%)
    * Valor: Contoso Wireless Laser Mouse E50 Grey | Frecuencia: 1 | Probabilidad: 0.0006 (0.06%)
    * Valor: Contoso Wireless Notebook Optical Mouse X205 Black | Frecu

In [48]:
#print("\n RESUMEN DE LA ESTRUCTURA DE LAS TABLAS")
#for nombre, dfo in tablas:
#    filas, columnas = dfo.shape
#    faltantes = dfo.isnull().sum().sum()
#    print(f" Tabla '{nombre}': {filas} filas, {columnas} columnas. Celdas faltantes totales: {faltantes}")

In [6]:
print(tablas[6]["GeographyKey"].head())

0    693
1    693
2    856
3    424
4    677
Name: GeographyKey, dtype: int64


In [79]:
print(tablas[7]["SalesQuantity"].describe())

count    2.282482e+06
mean     1.616679e+01
std      3.935030e+01
min      4.000000e+00
25%      9.000000e+00
50%      1.000000e+01
75%      1.300000e+01
max      2.880000e+03
Name: SalesQuantity, dtype: float64


In [8]:
#tablas[7]["SalesQuantity"].plot.kde()
#plt.xlim([0,50])

In [9]:
#tablas[6]["GeographyKey"].hist()

In [10]:
#df.head()

# **2. Transformación**

# **3. Carga y Análisis**